# MAPED on one CUDA GPU

This workflow keeps all seven input tilts in exact encoded form on GPU0. MAPED computes alignment summaries directly from those encoded sources, merges automatically sized regions, and writes a globally scaled uint16 result. The full float32 output is never allocated. The saved result reopens packed for live `Show4DSTEM` viewing. Set `MAPED_DATA_DIR` to the directory containing the seven `*_master.h5` files; optionally set `MAPED_OUTPUT` to choose the saved result path.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
from time import perf_counter
import torch
from quantem.diffraction import MAPEDTorch

try:
    SESSION = Path(os.environ["MAPED_DATA_DIR"]).expanduser().resolve()
except KeyError as exc:
    raise RuntimeError("Set MAPED_DATA_DIR to the directory containing seven *_master.h5 files.") from exc
FILES = sorted(SESSION.glob("*_master.h5"))
if len(FILES) != 7:
    raise ValueError(f"Expected seven *_master.h5 files in {SESSION}, found {len(FILES)}.")
OUTPUT = Path(os.environ.get("MAPED_OUTPUT", "maped-output/merged_master.h5")).expanduser().resolve()
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print(torch.cuda.get_device_name(0), "|", len(FILES), "tilts")


In [ ]:
started = perf_counter()
maped = MAPEDTorch.from_files(FILES, device="cuda:0")
torch.cuda.synchronize()
sources = maped.datasets.sources
assert len(sources) == len(FILES) == 7
assert all(source.representation.value == "encoded" for source in sources)
assert all(source.metadata["source_read_passes"] == 1 for source in sources)
print(f"Encoded resident ready: {sum(x.resident_bytes for x in sources) / 2**30:.2f} GiB in {perf_counter() - started:.2f} s")


In [ ]:
started = perf_counter()
maped.preprocess(plot_summary=False)
maped.diffraction_origin(sigma=1, plot_origins=False)
maped.diffraction_align(edge_blend=2, plot_aligned=False)
maped.real_space_align(
    num_iter=20, hanning_filter=True, padding=2, edge_blend=5,
    pad_val="median", shift_method="bilinear", plot_aligned=False,
)
torch.cuda.synchronize()
print(f"Alignment: {perf_counter() - started:.2f} s")

## Inspect before saving

Merge a small region with the complete detector and float32 intensities. All seven inputs remain available for another region or the full export below. Coordinates are `(row_start, row_stop, column_start, column_stop)`, with exclusive stops.


In [ ]:
started = perf_counter()
patch = maped.merge_datasets(
    scan_region=(252, 260, 252, 260), plot_result=False,
)
torch.cuda.synchronize()
print(f"Selected region: {perf_counter() - started:.3f} s")
patch_viewer = maped.show()
patch_viewer


## Save the complete merged result

This optional step computes one global uint16 scale, saves bounded regions, and reopens the complete result packed for viewing. It releases MAPED-owned inputs before reopening.


In [ ]:
started = perf_counter()
merged = maped.merge_datasets(save_to=OUTPUT, plot_result=False)
torch.cuda.synchronize()
precision = merged.metadata["precision"]
print(f"Merge + save + reopen: {perf_counter() - started:.2f} s")
print(f"scaled uint16 | RMSE {precision['rmse']:.4g} | max error {precision['max_abs_error']:.4g} | clipped {precision['clipped']}")

In [ ]:
viewer = maped.show()
viewer

Inspection retains the seven encoded inputs. Full saving releases those owned inputs before reopening the packed result. Close viewers before releasing the remaining resources:

```python
patch_viewer.close()
viewer.close()
maped.close()
```


## Processing time on CUDA GPU0

On RTX PRO 6000 Blackwell, preparation after loading took **1.66–1.68 s**. A selected DP took **14–16 ms** to merge; an 8×8 region took **0.10–0.12 s**. A separate full merge/BF/mean-DP/global-range benchmark took **5.72–5.92 s**, excluding export and rendering. Two complete float32 parity audits were exact. Loading measured **8.46–13.14 s**, and the processing/parity process peak remained below **13.3 GiB**.

Full packed scaled-uint16 viewing still uses the optional export above. The one-pass overview timing is not the time to prepare that entire resident view. See [processing qualification](../../docs/development/cuda-maped-processing-performance.md) for boundaries and retained evidence.

![Three diffraction patterns from the packed merged output](selected_diffraction_patterns.png)
